[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C24_Inference_Serving_Course/05_fp8_serving/05_fp8_serving.ipynb)

# 05 · fp8 与量化服务（用 numpy 做 cast + 误差账本）

目标：把 **浮点的指数/尾数解剖**、**fp8（E4M3/E5M2）cast**、**INT8 对称量化与误差界**、**scale 粒度（tensor/channel/token）**、**离群值灾难**、**KV 量化对注意力输出的影响** 用 numpy 实现出来，并用 `assert` 钉死每个数值不变量。

路线：浮点解剖 → fp8 cast 模拟 → INT8 对称量化 → scale 粒度对比 → 离群值实验 → KV 量化误差账 → ✏️ 练习 → 📖 答案 → 🧪 真实权重胶囊。

> 心智模型：**量化 = 缩放(scale) + 取整到一个网格 + 反量化**。INT8 是均匀网格(误差 ≤ scale/2)，FP8 是非均匀网格(误差 ∝ 数值)。scale 越被离群值撑大，正常值越受害。我们写的是*数值机制与误差*，不是吞吐。

## 1 · 浮点解剖：指数管范围，尾数管精度

任何浮点数 `= (-1)^s × 2^(e-bias) × 1.m`。**指数位定动态范围，尾数位定相对精度**。
用 numpy 直接读 float16/float32 的位，验证：相邻可表示数的间距 ≈ `2^(exp-mant)`（量级越大间距越大）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
# fp8/饱和实验里会出现「先算出超大中间值再钳到 max」的情形, 那只是被丢弃的临时溢出;
# 数学结果不受影响(超界值统一饱和到 max). 关掉 overflow 提醒让输出干净。
np.seterr(over='ignore')

def ulp(x, dtype):
    '''x 处的最小可表示间距(ULP)：下一个可表示数 - x。'''
    x = np.array(x, dtype=dtype)
    nxt = np.nextafter(x, np.array(np.inf, dtype=dtype))
    return float(nxt - x)

# float16: 10 尾数位 -> 量级 2^n 处间距 ~ 2^(n-10)
for v in [1.0, 2.0, 1024.0]:
    step16 = ulp(v, np.float16)
    n = np.floor(np.log2(v))
    print(f'fp16 在 {v:7.1f} 处间距={step16:.3e}  ≈ 2^(n-10)={2**(n-10):.3e}')
    assert abs(step16 - 2**(n-10)) < 1e-9, '间距应≈2^(指数-尾数位)'

# 动态范围对比：fp16 最大值 vs fp32 最大值
print(f'\nfp16 max = {np.finfo(np.float16).max}  (指数 5 位 -> 范围小)')
print(f'fp32 max = {np.finfo(np.float32).max:.3e}  (指数 8 位 -> 范围大)')
assert np.finfo(np.float16).max < 70000           # fp16 易溢出
assert np.finfo(np.float32).max > 1e38
print('✅ 间距∝2^(指数-尾数)：浮点保的是相对精度——大数处刻度稀疏(离群值的隐患)')

## 2 · fp8 cast 模拟：E4M3 / E5M2 的量化-反量化

fp8 是把数四舍五入到它那套**非均匀网格**上的最近可表示值，超过最大值就**饱和**。
实现思路：取量级 `e = floor(log2|x|)`，该量级步长 `step = 2^(e-mant)`，`round(|x|/step)*step`，最后钳到 `max_val`。
真实常数：**E4M3 max=448**、**E5M2 max=57344**。

In [ ]:
def fp8_cast(x, exp_bits, mant_bits, max_val):
    '''把 x 量化-反量化到 fp8 网格(round-to-nearest + 饱和)。返回 fp8 可表示的近似值。'''
    x = np.asarray(x, dtype=np.float64)
    sign = np.sign(x)
    ax = np.minimum(np.abs(x), max_val)             # 先饱和到 max(超界的统一钳住)
    bias = 2 ** (exp_bits - 1) - 1
    emin = 1 - bias                                  # 最小正规指数(次正规共用此量级)
    nz = ax > 0
    e = np.floor(np.log2(np.where(nz, ax, 1.0)))     # 量级
    e = np.clip(e, emin, None)
    step = np.exp2(e - mant_bits)                    # 该量级的网格步长
    q = np.round(ax / step) * step                  # 取整到最近网格点
    q = np.minimum(q, max_val)                      # 饱和到 max
    return sign * q

E4M3 = dict(exp_bits=4, mant_bits=3, max_val=448.0)
E5M2 = dict(exp_bits=5, mant_bits=2, max_val=57344.0)

# (1) 网格上的数无损往返：1.0, 1.5(=1.10b), 2.0 都可表示
for v in [0.0, 1.0, 1.5, 2.0]:
    assert abs(float(fp8_cast(v, **E4M3)) - v) < 1e-12, f'{v} 应无损'
# (2) 超过 max 饱和
assert float(fp8_cast(1e5, **E4M3)) == 448.0
assert float(fp8_cast(1e6, **E5M2)) == 57344.0
# (3) 最大可表示值就是 max 本身
assert float(fp8_cast(448.0, **E4M3)) == 448.0
print('E4M3 cast(0,1,1.5,2) 无损 ✅；cast(1e5)=448 饱和 ✅')
print('E5M2 cast(1e6)=57344 饱和 ✅')
print('✅ fp8 cast：网格内无损、超界饱和、max 符合真实常数(448 / 57344)')

再验证 fp8 的**误差是相对的**（∝数值），且严格落在半个步长内——这是非均匀网格的标志，也是 fp8 对大动态范围数据友好的原因。

In [ ]:
def fp8_step(x, exp_bits, mant_bits):
    ax = np.abs(np.asarray(x, float)); bias = 2**(exp_bits-1)-1; emin = 1-bias
    nz = ax > 0; e = np.floor(np.log2(np.where(nz, ax, 1.0))); e = np.clip(e, emin, None)
    return np.exp2(e - mant_bits)

x = rng.standard_normal(2000) * 3.0
x = np.clip(x, -440, 440)                            # 留在 E4M3 范围内
q = fp8_cast(x, **E4M3)
step = fp8_step(x, 4, 3)
err = np.abs(q - x)
assert np.all(err <= step/2 + 1e-9), 'fp8 误差应 ≤ 半个步长'
# 误差随数值大小增长(相对误差近似恒定) —— 非均匀网格特征
big = np.abs(x) > 10; small = (np.abs(x) > 0.1) & (np.abs(x) < 1)
print(f'大数(>10)平均绝对误差   = {err[big].mean():.4f}')
print(f'小数(0.1~1)平均绝对误差 = {err[small].mean():.4f}')
assert err[big].mean() > err[small].mean(), 'fp8 大数绝对误差更大(相对误差恒定)'
print('✅ fp8 误差 ≤ 半步长、且绝对误差∝数值 —— 非均匀网格，对大动态范围友好')

## 3 · INT8 对称量化：均匀网格，误差 ≤ scale/2

INT8 把张量映射到整数 `[-127,127]`：`scale=max|x|/127`，`q=round(x/scale)`，`x̂=q*scale`。
**均匀**网格(等间距 scale)，误差界 `|x̂-x| ≤ scale/2`。验证这条界，并看 scale 被最大值决定。

In [ ]:
def int8_quant(x):
    '''对称 INT8 量化-反量化。返回 (反量化值 x̂, scale).'''
    x = np.asarray(x, float)
    scale = np.abs(x).max() / 127.0
    if scale == 0: return x.copy(), 0.0
    q = np.round(x / scale)
    q = np.clip(q, -127, 127)
    return q * scale, scale

x = rng.standard_normal(1000) * 2.0
xhat, scale = int8_quant(x)
err = np.abs(xhat - x)
print(f'scale = max|x|/127 = {scale:.4f}')
print(f'最大量化误差 = {err.max():.4f}   半个 scale = {scale/2:.4f}')
assert np.all(err <= scale/2 + 1e-9), 'INT8 误差应 ≤ scale/2'
# scale 由最大值决定：把一个数变大 -> scale 变大 -> 所有数误差上界变大
x2 = x.copy(); x2[0] = 50.0                          # 放一个较大的数
_, scale2 = int8_quant(x2)
assert scale2 > scale * 5, '最大值变大 -> scale 跟着被撑大'
print(f'把一个数改成 50 后 scale={scale2:.3f} (放大 {scale2/scale:.0f}x) —— 最大值绑架全局 scale')
print('✅ INT8 均匀网格，误差≤scale/2；scale 由 max|x| 决定(离群值灾难的伏笔)')

## 4 · scale 粒度：per-tensor vs per-channel vs per-token

把 scale 切细，离群值只连累它所在的小组。对同一个带离群的 2D 张量 `[tokens, channels]`，对比三种粒度的量化误差。
约定：**per-token** = 每行一个 scale，**per-channel** = 每列一个 scale。

In [ ]:
def int8_quant_axis(x, axis):
    '''沿给定 axis 分组、每组一个 scale 的对称 INT8。axis=1 -> per-token(每行); axis=0 -> per-channel(每列).'''
    x = np.asarray(x, float)
    amax = np.abs(x).max(axis=axis, keepdims=True)
    scale = np.where(amax == 0, 1.0, amax / 127.0)
    q = np.clip(np.round(x / scale), -127, 127)
    return q * scale

def rel_err(xhat, x, mask=None):
    num = np.abs(xhat - x); den = np.abs(x)
    if mask is not None: num, den = num[mask], den[mask]
    return num.sum() / den.sum()

# 构造 [8 tokens, 16 channels]：大多 N(0,1)，但第 0 列是离群通道(~80)
X = rng.standard_normal((8, 16))
X[:, 0] = 80.0                                       # 离群通道
normal = np.ones_like(X, bool); normal[:, 0] = False  # 正常元素掩码

xt, _   = int8_quant(X)                              # per-tensor
xc      = int8_quant_axis(X, axis=0)                 # per-channel(每列)
xk      = int8_quant_axis(X, axis=1)                 # per-token(每行)
e_tensor  = rel_err(xt, X, normal)
e_channel = rel_err(xc, X, normal)
print(f'正常元素相对误差: per-tensor={e_tensor:.3f}  per-channel={e_channel:.4f}')
assert e_channel < e_tensor / 5, 'per-channel 应把离群通道隔离, 正常元素误差大幅下降'
# 离群在「列」上 -> per-channel 隔离最干净; per-token 每行都含那个离群列, 改善有限
print('✅ scale 越细, 离群污染半径越小：per-channel 把离群通道关进单独的笼子')

## 5 · 离群值实验：复现 LLM.int8() 的灾难

Dettmers et al. 2022：LLM 激活有少数维度比其余大一两个数量级。per-tensor 量化下，这些离群值把 scale 撑大，**正常值精度崩塌**。
我们对比「有离群」vs「无离群」时，**正常维度**的相对误差差多少。

In [ ]:
D = 256
base = rng.standard_normal(D)                       # 正常激活 ~ N(0,1)
normal_mask = np.ones(D, bool)
outlier_dims = [7, 42, 113]                         # 少数离群维
normal_mask[outlier_dims] = False

# 有离群：少数维 ~100
x_out = base.copy(); x_out[outlier_dims] = 100.0
xhat_out, s_out = int8_quant(x_out)
rel_with = rel_err(xhat_out, x_out, normal_mask)    # 正常维的相对误差

# 无离群：那几维换成正常值
x_clean = base.copy(); x_clean[outlier_dims] = rng.standard_normal(3)
xhat_clean, s_clean = int8_quant(x_clean)
rel_without = rel_err(xhat_clean, x_clean, normal_mask)

print(f'scale: 有离群={s_out:.3f}  无离群={s_clean:.4f}  (离群把 scale 撑大 {s_out/s_clean:.0f}x)')
print(f'正常维相对误差: 有离群={rel_with:.3f}  无离群={rel_without:.4f}  (恶化 {rel_with/rel_without:.0f}x)')
assert rel_with > rel_without * 5, '离群值应让正常维误差恶化一个数量级以上'
print('✅ 坐实 LLM.int8() 的发现：极少数离群值通过 per-tensor scale 毁掉绝大多数正常值的精度')

## 6 · KV 量化误差账：传导到注意力输出

把 KV 存成 INT8/FP8 省显存与带宽，代价是注意力输出的偏差。好消息：KV 误差经 softmax 加权平均后会被**部分抵消**，输出偏差通常远小于 KV 的逐元素误差。
验证：把 K、V 量化后算注意力，对比 FP32，且 **fp8 偏差 < int8?** 不一定——这里看 **更细 scale(per-token) 比 per-tensor 偏差更小**。

In [ ]:
def attention(Q, K, V):
    d = Q.shape[-1]
    s = Q @ K.T / np.sqrt(d)
    s = s - s.max(axis=-1, keepdims=True)
    w = np.exp(s); w = w / w.sum(axis=-1, keepdims=True)
    return w @ V

n, d = 6, 8
Q = rng.standard_normal((n, d))
K = rng.standard_normal((n, d))
V = rng.standard_normal((n, d))
O_fp32 = attention(Q, K, V)                          # 参考

# per-tensor INT8 量化 K,V
Kt, _ = int8_quant(K); Vt, _ = int8_quant(V)
O_tensor = attention(Q, Kt, Vt)
# per-token INT8 (每行一个 scale, 更细)
Kk = int8_quant_axis(K, axis=1); Vk = int8_quant_axis(V, axis=1)
O_token = attention(Q, Kk, Vk)

err_tensor = np.abs(O_tensor - O_fp32).mean()
err_token  = np.abs(O_token  - O_fp32).mean()
kv_elem_err = np.abs(Kt - K).mean()
print(f'KV 逐元素平均误差(per-tensor) = {kv_elem_err:.4f}')
print(f'注意力输出误差: per-tensor={err_tensor:.4f}  per-token={err_token:.4f}')
assert err_tensor < kv_elem_err, '输出误差应小于 KV 逐元素误差(softmax 平均掉一部分)'
assert err_token <= err_tensor + 1e-9, 'per-token 更细 -> 输出偏差不更差'
print('✅ KV 量化误差经注意力被部分平均；更细的 scale 粒度让输出偏差更小')

---
## ✏️ 练习 1：通用 fp8 cast

实现 `cast_to_grid(x, mant_bits, max_val)`：把标量/数组 `x` 量化到尾数 `mant_bits` 位的浮点网格(round-to-nearest)、并饱和到 `max_val`。
(就是 worked 2 的 fp8_cast，但用 `emin` 由你按惯例取很小的量级即可——这里固定 `emin=-14` 简化。)

In [ ]:
def cast_to_grid(x, mant_bits, max_val, emin=-14):
    # TODO: ax=|x|; e=clip(floor(log2(ax)), emin, None); step=2^(e-mant_bits)
    #       q=round(ax/step)*step; 钳到 max_val; 乘回符号
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 网格内无损
for v in [1.0, 1.5, 2.0, 3.0]:
    assert abs(float(cast_to_grid(v, mant_bits=3, max_val=448.0)) - v) < 1e-9
# 饱和
assert float(cast_to_grid(1e5, mant_bits=3, max_val=448.0)) == 448.0
assert float(cast_to_grid(-1e5, mant_bits=2, max_val=57344.0)) == -57344.0
# 误差 ≤ 半步长
xs = rng.standard_normal(500) * 2.0
q = cast_to_grid(xs, mant_bits=3, max_val=448.0)
e = np.floor(np.log2(np.clip(np.abs(xs), 2.0**-14, None)))
step = 2.0 ** (e - 3)
assert np.all(np.abs(q - xs) <= step/2 + 1e-9)
print('✅ 练习 1 通过：fp8 cast 网格内无损、超界饱和、误差≤半步长')

## ✏️ 练习 2：per-token scale 选择

实现 `per_token_int8(X)`：对 `X[tokens, dim]` 做 **per-token**(每行一个 scale) 对称 INT8 量化-反量化，返回 `(X̂, scales)`，`scales` 形状 `[tokens, 1]`。
要求每行的 `scale = max|该行|/127`(全 0 行 scale 取 1 防除零)。

In [ ]:
def per_token_int8(X):
    # TODO: 每行 scale = max|行|/127 (全0行取1); q=clip(round(X/scale),-127,127); 返回 (q*scale, scale)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
X = rng.standard_normal((5, 12)); X[2] *= 30        # 第2行整体偏大
Xhat, scales = per_token_int8(X)
assert scales.shape == (5, 1)
# 每行误差 ≤ 该行 scale/2
assert np.all(np.abs(Xhat - X) <= scales/2 + 1e-9)
# 大行有更大的 scale, 但各行相对误差量级相近(per-token 隔离了行间差异)
assert scales[2,0] > scales[0,0], '偏大的行 scale 更大'
rel = np.abs(Xhat - X).sum(1) / np.abs(X).sum(1)
assert rel.max() < 0.05, 'per-token 下每行相对误差都很小'
print('✅ 练习 2 通过：per-token 每行独立 scale, 行间差异被隔离')

## ✏️ 练习 3：离群值对 per-tensor scale 的连累

实现 `scale_inflation(x, outlier_val)`：把数组 `x` 的第 0 个元素替换成 `outlier_val`，返回
`(原 per-tensor scale, 注入离群后的 scale, 放大倍数)`。用它量化「离群值把 scale 撑大多少倍」。

In [ ]:
def scale_inflation(x, outlier_val):
    # TODO: s0 = max|x|/127; x2 = x.copy(); x2[0]=outlier_val; s1 = max|x2|/127
    #       返回 (s0, s1, s1/s0)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
x = rng.standard_normal(100)                        # 大多在 [-3,3]
s0, s1, ratio = scale_inflation(x, outlier_val=100.0)
assert s1 > s0, '注入离群值后 scale 变大'
assert ratio > 10, '一个 100 的离群值应把 scale 撑大 10 倍以上'
assert abs(s1 - 100.0/127) < 1e-9, '注入后 scale 由离群值 100 决定'
print(f'原 scale={s0:.4f} -> 注入离群后={s1:.4f} (放大 {ratio:.0f}x)')
print('✅ 练习 3 通过：单个离群值即可绑架全局 scale, 连累所有正常值')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cast_to_grid(x, mant_bits, max_val, emin=-14):
    x = np.asarray(x, float)
    sign = np.sign(x); ax = np.abs(x)
    nz = ax > 0
    e = np.floor(np.log2(np.where(nz, ax, 1.0)))
    e = np.clip(e, emin, None)
    step = np.exp2(e - mant_bits)
    q = np.round(ax / step) * step
    q = np.minimum(q, max_val)
    return sign * q

In [ ]:
# 练习 2 参考答案
def per_token_int8(X):
    X = np.asarray(X, float)
    amax = np.abs(X).max(axis=1, keepdims=True)
    scale = np.where(amax == 0, 1.0, amax / 127.0)
    q = np.clip(np.round(X / scale), -127, 127)
    return q * scale, scale

In [ ]:
# 练习 3 参考答案
def scale_inflation(x, outlier_val):
    x = np.asarray(x, float)
    s0 = np.abs(x).max() / 127.0
    x2 = x.copy(); x2[0] = outlier_val
    s1 = np.abs(x2).max() / 127.0
    return s0, s1, s1 / s0

---
## 🧪 真实数据胶囊：量化真实 GPT-2 权重

用**真实**的 GPT-2 权重(若本机/联网可取则用 `transformers` 加载一层权重；否则回退到一组真实量级的内置数值)，做 per-tensor INT8 与 fp8(E4M3) 量化，测量真实权重上的量化误差——把数值机制接到真实模型。

> 真实权重大多集中在 `[-0.5, 0.5]`、含少量较大值，正是量化要对付的分布。

In [ ]:
# 取一层真实 GPT-2 权重；联网/缓存不可用则回退到真实量级的内置样本(保证离线可跑)
def load_real_weights():
    try:
        from transformers import GPT2Model
        m = GPT2Model.from_pretrained('gpt2')
        W = m.h[0].mlp.c_fc.weight.detach().numpy().astype(np.float64).ravel()
        return W[:4096], 'gpt2 c_fc (真实加载)'
    except Exception as e:
        # 离线回退：一组贴近真实 GPT-2 权重统计(均值~0、std~0.13、少量较大值)的样本
        r = np.random.default_rng(1234)
        W = r.normal(0.0, 0.13, 4096)
        W[::500] *= 6.0                              # 少量较大权重(真实权重也有)
        return W, 'offline fallback (真实量级统计)'

W, src = load_real_weights()
print(f'权重来源: {src};  n={W.size}, std={W.std():.4f}, max|w|={np.abs(W).max():.3f}')

**🧪 胶囊练习**：实现 `quant_error_report(W)`：对真实权重 `W` 分别做 per-tensor INT8 与 E4M3 量化，返回两者的**均方根误差(RMSE)** `(rmse_int8, rmse_fp8)`。复用上面的 `int8_quant` 与 `fp8_cast`。

In [ ]:
def quant_error_report(W):
    # TODO: int8: xhat,_ = int8_quant(W); fp8: xhat = fp8_cast(W, **E4M3)
    #       rmse = sqrt(mean((xhat-W)^2)); 返回 (rmse_int8, rmse_fp8)
    raise NotImplementedError

In [ ]:
# 自测
rmse_int8, rmse_fp8 = quant_error_report(W)
print(f'真实权重量化 RMSE: INT8={rmse_int8:.5f}  E4M3={rmse_fp8:.5f}')
assert rmse_int8 >= 0 and rmse_fp8 >= 0
# 两者都应远小于权重本身的尺度(量化是近似而非乱来)
assert rmse_int8 < W.std(), 'INT8 量化误差应远小于权重 std'
assert rmse_fp8 < W.std(), 'fp8 量化误差应远小于权重 std'
print('✅ 胶囊练习通过：真实权重上 INT8 与 fp8 量化误差都远小于权重尺度')

In [ ]:
# 📖 胶囊参考答案
def quant_error_report(W):
    W = np.asarray(W, float)
    xhat_int8, _ = int8_quant(W)
    xhat_fp8 = fp8_cast(W, **E4M3)
    rmse_int8 = float(np.sqrt(np.mean((xhat_int8 - W) ** 2)))
    rmse_fp8 = float(np.sqrt(np.mean((xhat_fp8 - W) ** 2)))
    return rmse_int8, rmse_fp8

---
### 小结
- **decode 受带宽支配**，量化把每个数的字节砍半 → decode 近乎提速一倍，且不改调度。
- 浮点 = 符号 × 2^指数 × 1.尾数：**指数管范围、尾数管精度**；fp8 两种配方 **E4M3**(精度,max=448) / **E5M2**(范围,max=57344)。
- 量化 = scale + 取整：INT8 均匀网格(误差≤scale/2)，fp8 非均匀网格(误差∝数值)。
- **scale 粒度** per-tensor→channel→token 越细越抗离群；**离群值**把 per-tensor scale 撑大、毁掉正常值精度(LLM.int8())。
- **KV 量化**省显存+带宽，误差经 softmax 被部分平均；与模块 01 分页正交可叠加。

### 🎓 全课结语：六个模块，一座推理引擎
你已经从零拆开并亲手实现了现代 LLM 推理引擎的**五根支柱**：
1. **PagedAttention**(01) — KV 像虚拟内存般分页, 几乎消灭碎片, 让 batch 开得更大;
2. **Continuous Batching**(02) — 迭代级调度, 完成即让位/到达即补位, 把 GPU 喂满;
3. **投机解码**(03) — draft 猜 + target 并行验证 + 接受/拒绝采样, 无损减少串行步数;
4. **Prefix Cache 与 PD 分离**(04) — radix 树自动复用前缀, prefill/decode 分到不同 GPU 互不干扰;
5. **量化**(05) — fp8/INT8 把每步搬运的字节砍半, 直击带宽受限的 decode。

它们都从同一张「**prefill 算力受限 / decode 带宽受限**」的两阶段瓶颈表推导而来, 攻击推理不同侧面的浪费, 组合起来就是 vLLM / SGLang / TensorRT-LLM 让吞吐提升一个数量级的全部秘密。

你在 numpy 里验证过的每个机制(块表寻址、CoW、迭代调度、接受采样、radix 匹配、fp8 cast), 都能一对一对回真实引擎的源码。**下一步**: 打开 vLLM / SGLang 的代码, 你会发现里面全是老朋友。